# Day 017：Tokenizer 训练脚本与产物

本 Notebook 使用仓库已有 tokenizer 做验收，不重新训练词表。训练脚本明确提醒：新 tokenizer 与旧模型权重不兼容。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('../../minimind/model')
print(type(tokenizer).__name__)
print('vocab size:', len(tokenizer))
print('bos/eos/pad:', tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id)
print('has chat template:', bool(tokenizer.chat_template))

## 1. 特殊 token

基础边界 token 在 `skip_special_tokens=True` 时隐藏；`<think>`、`<tool_call>` 等控制标记保持可见，方便外层解析。

In [ ]:
for text in ['<|endoftext|>', '<|im_start|>', '<think>', '<tool_call>', '<|buffer1|>']:
    token_id = tokenizer.convert_tokens_to_ids(text)
    print(text, '->', token_id, '| skip=True:', repr(tokenizer.decode([token_id], skip_special_tokens=True)))

## 2. Encode/decode 可逆性

关闭 `skip_special_tokens`，否则边界标记会被删除，无法与原始 prompt 比较。

In [ ]:
messages = [
    {'role': 'user', 'content': '你来自哪里？'},
    {'role': 'assistant', 'content': '我来自地球'}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False)
ids = tokenizer(prompt)['input_ids']
decoded = tokenizer.decode(ids, skip_special_tokens=False)
print(prompt)
print('token count:', len(ids))
print('round trip:', decoded == prompt)

## 3. 粗略压缩率

字符数除以 token 数，只能作为不同文本和 tokenizer 的粗略比较。

In [ ]:
texts = ['人工智能正在改变很多行业。', 'Large language models predict the next token.']
for text in texts:
    count = len(tokenizer.encode(text))
    print({'chars': len(text), 'tokens': count, 'chars_per_token': len(text) / count})